In [4]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
import pandas as pd

In [5]:
df = pd.read_csv(r"..\dataset\dataset_module_one_DM2.csv")

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report
# Use imblearn Pipeline to properly handle resampling during cross-validation
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

# 1. Split features and target
X = df.drop(columns=['sii'])
y = df['sii']

# 2. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 3. Setup SMOTE
smote_strategy = {
    1: 2000,  # Intermediate 1
    2: 2000,  # Intermediate 2
    3: 1500   # Minority class
}
smote = SMOTE(sampling_strategy=smote_strategy, random_state=42)

# 4. Create imblearn Pipeline (SMOTE + Scaling + Model)
pipeline = Pipeline([
    ('smote', smote),
    ('scaler', StandardScaler()),
    ('svm', SVC(class_weight='balanced'))
])

# 5. Define Hyperparameters 
param_grid = {
    'svm__C': [1, 5, 10, 15, 20, 30],
    'svm__kernel': ['linear', 'rbf']
}

# 6. Grid Search Setup 
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=2
)

# 7. Execute Training
print("Starting Cross Validation with SMOTE and scaling...")
grid_search.fit(X_train, y_train)

# 8. Print Results
print("\n--- CV RESULTS ---")
print("Best Params:", grid_search.best_params_)
print("Best CV F1-Macro:", round(grid_search.best_score_, 4))

# 9. Evaluate on Test Set
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print("\n--- TEST SET PERFORMANCE ---")
print(classification_report(y_test, y_pred))

Starting Cross Validation with SMOTE and scaling...
Fitting 5 folds for each of 12 candidates, totalling 60 fits

--- CV RESULTS ---
Best Params: {'svm__C': 30, 'svm__kernel': 'rbf'}
Best CV F1-Macro: 0.3228

--- TEST SET PERFORMANCE ---
              precision    recall  f1-score   support

         0.0       0.74      0.75      0.75      1746
         1.0       0.26      0.22      0.24       476
         2.0       0.23      0.21      0.22       269
         3.0       0.02      0.08      0.03        25

    accuracy                           0.59      2516
   macro avg       0.31      0.32      0.31      2516
weighted avg       0.59      0.59      0.59      2516

